---
numbering: false
---

# 0.2 Derivatives, Sensitivities, and Local Optimality

Calculus describes rates of change and sensitivities. In CCD, a derivative may be a physical rate such as velocity, a linearization of nonlinear dynamics, or the sensitivity of performance to a plant or controller decision.

This section reviews the derivative rules, extrema, curvature, and smoothness needed for dynamic modeling and gradient-based optimization.

## Tangent Lines

Suppose $f$ is a function that takes in a single real number and outputs a single real number, i.e. $f: \mathbb{R} \to \mathbb{R}$.

If $f$ is a function, then the **derivative** of $f$ is another function, sometimes denoted $f'$, such that $f'(x)$ is the "instantaneous" rate of change of $f$ at the input $x$.

To understand what I mean by "instantaneous" rate of change, let's consider an example. Suppose $f(x) = \frac{1}{4}x^2 - 3$. The graph of $f$ is shown below, in <span style="color: #3d81f6; font-weight: bold;">blue</span>, along with a slider for input values of $x$. Drag the slider.

In [ ]:
from utils import plot_function_with_tangent_line

f = lambda x: (1/4)*x**2 - 3
f_prime = lambda x: (1/2)*x
x_range = (-6, 6)
y_range = (-6, 6)

fig = plot_function_with_tangent_line(f, f_prime, x_range, y_range)
fig.update_layout(width=600, height=500)
fig.show(renderer='notebook')

At any point $x$, the <span style="color:orange; font-weight: bold;">tangent line</span> to the graph of $f$ is the **best linear approximation** of the graph of $f$ near $x$.

For instance, consider when $x=-4$. At $x = -4$, $f(-4) = \frac{1}{4}(-4)^2 - 3 = 1$.

In [ ]:
from utils import plot_functions
import numpy as np

# Define the function and its derivative
f = lambda x: (1/4)*x**2 - 3
f_prime = lambda x: (1/2)*x

# Create the plot with both the function and tangent line at x = -4
x_range = (-6, 6)
y_range = (-6, 6)

# Calculate tangent line points
x_point = -4
y_point = f(x_point)
slope = f_prime(x_point)

# Create points for tangent line (extending 2 units in each direction)
x_tangent = [x_point - 0.5, x_point + 0.5]
y_tangent = [y_point + slope * (-0.5), y_point + slope * 0.5]

# Plot both the function and tangent line
fig = plot_functions(
    [f, lambda x: np.where((-5.34 < x) & (x < -2.65), slope * (x - x_point) + y_point, None)],
    ['f(x)', 'Tangent line'],
    [x_point],
    x_range=x_range,
    y_range=y_range
)
fig.update_layout(width=600, height=450, title="Tangent line at x = -4")
fig.show(renderer='notebook')

The tangent line at $x=-4$ is the line that passes through the point $(-4, 1)$ that best approximates $f$ near $x=-4$, among all other lines that pass through $(-4, 1)$. In the plot above, use your mouse to zoom in on the point $(-4, 1)$, and you'll see that when zoomed in, the <span style="color:orange; font-weight: bold;">tangent line</span> and <span style="color: #3d81f6; font-weight: bold;">original function</span> are very difficult to distinguish.

:::{attention} The Derivative is the Slope of the Tangent Line

Here's the million dollar connection. The slope of the tangent line at $x$ **is, by definition**, the derivative of $f$ at $x$.

In the example above, when $x = -4$, the slope of the tangent line is $-2$, which is the derivative of $f$ at $x=-4$, i.e. $f'(-4) = -2$.
:::

This intuitive definition of the derivative – as being the slope of the tangent line – is the most important way to think about the derivative. The formal definition is also important, and we'll get there next, but you should have enough context for this first activity.

:::{tip} Activity 0.2.1
:class: dropdown
:name: derivatives-activity-1
Let $f(x) = \frac{1}{4}x^4 + \frac{1}{3}x^3 - x^2 + 2$.

Given that $f'(x)=x^3+x^2-2x$, find the equation of the tangent line to $f(x)$ at the following points:

- $x=-3$
- $x=-1$
- $x=1$

As a bonus exercise, try to verify that the provided formula for $f'(x)$ is correct, using your knowledge of derivatives from Calculus 1.
:::

---

## Formal Definition

### Secant Lines

Let's review the more formal definition of the derivative. First, remember the general formula for the slope of the line between two points $(x_1, y_1)$ and $(x_2, y_2)$: $$\frac{y_2 - y_1}{x_2 - x_1}$$

Let's say we're trying to find the slope of the tangent line at $x=a$, which is the line that passes through the point $(a, f(a))$ whose slope is the instantaneous rate of change of $f$ at $x=a$. To find that instantaneous rate of change, we can find the slope of the line between $(a, f(a))$ and some other point $(b, f(b))$, where $b-a$ is as close to $0$ as possible.

In the example below, as $b$ approaches $a = 1$, the slope of the line between $(1, f(1))$ and $(b, f(b))$ approaches the slope of the tangent line at $x=1$. Note that the formal name for the line between any two points on a function is a **secant line**.

In [ ]:
from utils import plot_multiple_secant_lines

f = lambda x: (1/4)*x**2 - 3
x_range = (-6, 6)
y_range = (-6, 6)

# Define the three secant line pairs
secant_points = [
    (1, 5),      # Secant between 1 and 5
    (1, 2),     # Secant between 1 and 2 
    (1, 1.1)   # Secant between 1 and 2.1
]

fig = plot_multiple_secant_lines(f, secant_points, x_range, y_range)
fig.update_layout(width=700, height=300)
fig.show(renderer='notebook')

### Limits

Let's be more precise, using the idea of a **limit** from Calculus 1. Recall, $\displaystyle \lim_{x \rightarrow a} g(x) = L$ is pronounced "the limit of $g(x)$ as $x$ approaches $a$ is $L$". If $\displaystyle \lim_{x \rightarrow a} g(x) = L$, then as $x$ gets closer and closer to $a$, $g(x)$ gets closer and closer to $L$. (Intuitively, you might think this must always mean that $g(a) = L$, but that's not always the case.)

The slope of the tangent line at $x=a$ is the limit of the slope of the line between $(a, f(a))$ and $(b, f(b))$ as $b$ approaches $a$:

$$\text{slope of tangent line at } a = \lim_{b \to a}\frac{f(b)-f(a)}{b-a}$$

This definition alone can help us compute some common derivatives. For instance, if $f(x) = x^2$, then the slope of the tangent line at $x=a$ is:

$$\lim_{b \to a}\frac{f(b)-f(a)}{b-a} = \lim_{b \to a}\frac{b^2-a^2}{b-a} = \lim_{b \to a}\frac{(b-a)(b+a)}{b-a} = \lim_{b \to a}(b+a) = 2a$$

Here, I used the difference of squares formula, $b^2 - a^2 = (b-a)(b+a)$. To find the slope of the tangent line using the limit definition, you'll need to use some sort of algebraic manipulation to simplify the expression, because the limit of the denominator is 0, and we can't divide by 0.

Instead of thinking of the slope of the tangent line as the limit of the slope of the line between the points $(a, f(a))$ and $(b, f(b))$ as $b \rightarrow a$, a more general (and equivalent!) definition of the derivative is the limit of the slope of the line between the points $(x, f(x))$ and $(x+h, f(x+h))$ as $h \rightarrow 0$.

### Derivatives

:::{note} Definition: Derivative

Suppose $f: \mathbb{R} \to \mathbb{R}$ is a function. Then the **derivative** of $f$ at $x$ is defined as:

$$\frac{\text{d}f}{\text{d}x}(x) = \lim_{h \to 0}\frac{f(x+h)-f(x)}{h}$$

This is pronounced "the derivative of $f$ with respect to $x$."
:::

We will not repeatedly use this limit definition, but it explains what numerical finite-difference derivatives approximate and why the perturbation size matters.

There are two equivalent notations for the derivative: $\frac{\text{d}f}{\text{d}x}(x)$ and $f'(x)$. I used the notation $f'(x)$ in the previous section since it's easier to write and more commonly used in calculus courses. However, I'll use the notation $\frac{\text{d}f}{\text{d}x}(x)$ from now on, as it'll make the transition to multivariable calculus more natural when we get there.

Often, for brevity, I will drop the $(x)$ and just write $\frac{\text{d}f}{\text{d}x}$. As an example, suppose $g(x) = \sin^2(x) + 3 \log (x)$, where $\log( \cdot )$ is the natural logarithm (with base $e$). Then, the derivative of $g$ is:

$$\frac{\text{d}g}{\text{d}x} = 2\sin(x)\cos(x) + \frac{3}{x}$$

$\frac{\text{d}g}{\text{d}x}$ (equivalently, $\frac{\text{d}g}{\text{d}x}(x)$) is a function, not a number. To get a number as an output, we need to plug in a value for $x$. For example, $\frac{\text{d}g}{\text{d}x}(\pi)$ is the number ($\frac{3}{\pi}$) corresponding to the slope of the tangent line to $g$ at $x=\pi$.

To actually find $\frac{\text{d}g}{\text{d}x}$, we used several derivative rules, which we'll now review.

---

## Rules and Examples

:::{note} Five Key Derivative Rules
1. **Constant Rule**: If $f(x) = c$, where $c$ is some constant, then:

    $$\frac{\text{d}f}{\text{d}x} = 0$$

1. **Addition Rule**: If $h(x) = f(x) + g(x)$, then:

    $$\frac{\text{d}h}{\text{d}x} = \frac{\text{d}f}{\text{d}x} + \frac{\text{d}g}{\text{d}x}$$

    Equivalently, $h'(x) = f'(x) + g'(x)$.

1. **Power Rule**:

    $$\frac{\text{d}}{\text{d}x}x^n = nx^{n-1}$$

1. **Product Rule**: If $h(x) = f(x)g(x)$, then:

    $$\frac{\text{d}h}{\text{d}x} = f \cdot \frac{\text{d}g}{\text{d}x} + \frac{\text{d}f}{\text{d}x} \cdot g$$

    Equivalently, $h'(x) = f(x) g'(x) + f'(x) g(x)$.

1. **Chain Rule**: If $h(x) = f(g(x))$, then:

    $$\frac{\text{d}h}{\text{d}x} = \frac{\text{d}f}{\text{d}g} \cdot \frac{\text{d}g}{\text{d}x}$$

    Equivalently, $h'(x) = f'(g(x)) g'(x)$.

:::

These rules can all be proved using the formal definition of the derivative, but our emphasis is their use in engineering models and optimization.

Attempt each of the following examples before peeking at their solutions.

### Example: Polynomials

Differentiate $f(x) = 4x^5 + 3x^2 + 2x + 1$.

:::{seealso} Solution
:class: dropdown

None of the five key rules directly specify how to take the derivative of an expression like $4x^5$, and even though you _may_ look at this expression and see that its derivative is $20 x^4$, it's well worth our time to review the rules carefully. 

Let's split $4x^5$ into two separate functions, $4$ and $x^5$, and then use the product rule.

$$\frac{\text{d}}{\text{d}x} 4x^5 = 4 \left( \frac{\text{d}}{\text{d}x} x^5 \right) + \left( \frac{\text{d}}{\text{d}x}4 \right) x^5 = 4 \cdot 5x^4 + 0 = 20x^4$$

While applying the product rule, I also used the power rule to differentiate $x^5$, and the constant rule to differentiate $4$.

With this in mind, we can return to the goal of differentiating $f(x)$.

$$
\begin{align*}
\frac{\text{d}f}{\text{d}x} &= \frac{\text{d}}{\text{d}x} (4x^5 + 3x^2 + 2x + 1) \\
&= \frac{\text{d}}{\text{d}x} 4x^5 + \frac{\text{d}}{\text{d}x} 3x^2 + \frac{\text{d}}{\text{d}x} 2x + \frac{\text{d}}{\text{d}x} 1 \\
&= 20x^4 + 6x + 2
\end{align*}
$$
:::

### Example: Reciprocals

Differentiate $f(x) = \frac{1}{x^2 - 1}$.

:::{seealso} Solution
:class: dropdown

Note that $f(x)$ can also be written as $f(x) = (x^2 - 1)^{-1}$, which looks like something we can use the power rule on. To fully apply the power rule, we'll need to use the chain rule, too.

$$
\begin{align*}
\frac{\text{d}}{\text{d}x} (x^2 - 1)^{-1} &= \underbrace{-1 \cdot (x^2 - 1)^{-2}}_{\text{power rule}} \:\:\:\: \cdot \underbrace{\frac{\text{d}}{\text{d}x} (x^2 - 1)}_{\text{required by the chain rule}} \\
&= -1 \cdot (x^2 - 1)^{-2} \cdot 2x \\
&= -\frac{2x}{(x^2 - 1)^2}
\end{align*}
$$

:::

### Example: Quotients

Differentiate $f(x) = \frac{x^2 + 1}{x^2 - 1}$.

:::{seealso} Solution
:class: dropdown

It looks like $f(x)$ is a quotient of two functions, $x^2 + 1$ and $x^2 - 1$, but I intentionally did not introduce a quotient rule – it's unnecessary, and can be replicated using the product rule and chain rule, since $f(x)$ can be written as a product:

$$
f(x) = \frac{x^2 + 1}{x^2 - 1} = (x^2 + 1) \cdot \frac{1}{x^2 - 1}
$$

You'll notice that we've already found the derivative of $\frac{1}{x^2 - 1}$ in Example 2, so we can use that result here while applying the product rule.

$$
\begin{align*}
\frac{\text{d}f}{\text{d}x} &= \frac{\text{d}}{\text{d}x} \left( (x^2 + 1) \cdot \frac{1}{x^2 - 1} \right) \\
&= \underbrace{\left( \frac{\text{d}}{\text{d}x} (x^2 + 1) \right) \cdot \frac{1}{x^2 - 1} + (x^2 + 1) \cdot \left( \frac{\text{d}}{\text{d}x} \frac{1}{x^2 - 1} \right)}_{\text{product rule}} \\
&= 2x \cdot \frac{1}{x^2 - 1} + (x^2 + 1) \cdot \underbrace{\left( -\frac{2x}{(x^2 - 1)^2} \right)}_{\text{from Example 2}} \\
&= \frac{2x}{x^2 - 1} - \frac{2x(x^2 + 1)}{(x^2 - 1)^2} \\
&= \underbrace{\frac{2x(x^2 - 1) - 2x(x^2 + 1)}{(x^2 - 1)^2}}_{\text{common denominator}} \\
&= \frac{2x(x^2 - 1 - x^2 - 1)}{(x^2 - 1)^2} \\
&= \frac{-4x}{(x^2 - 1)^2}
\end{align*}
$$
:::

### Example: Trigonometric and Logarithmic Functions

Differentiate $f(x) = \log(\cos(e^x))$.

To do so, you'll need to remember a few other common derivatives that the five key rules don't cover.

**Trigonometric Derivatives**: 

$$\frac{\text{d}}{\text{d}x} \sin(x) = \cos(x)$$
$$\frac{\text{d}}{\text{d}x} \cos(x) = -\sin(x)$$

**Logarithmic Derivatives**: If $\log( \cdot )$ is the natural logarithm (with base $e$), then:

$$\frac{\text{d}}{\text{d}x} \log(x) = \frac{1}{x}$$
$$\frac{\text{d}}{\text{d}x} \log_b(x) = \frac{1}{x \log(b)}$$

**Exponential Derivatives**:

$$\frac{\text{d}}{\text{d}x} e^x = e^x$$
$$\frac{\text{d}}{\text{d}x} a^x = a^x \log(a)$$

:::{seealso} Solution
:class: dropdown

Differentiating $f(x) = \log(\cos(e^x))$ requires a few careful applications of the chain rule.

$$
\begin{align*}
\frac{\text{d}f}{\text{d}x} &= \frac{\text{d}}{\text{d}x} \log(\cos(e^x)) \\
&= \frac{1}{\cos(e^x)} \cdot \frac{\text{d}}{\text{d}x} \cos(e^x) \\
&= \frac{1}{\cos(e^x)} \cdot (-\sin(e^x)) \cdot \frac{\text{d}}{\text{d}x} e^x \\
&= \frac{1}{\cos(e^x)} \cdot (-\sin(e^x)) \cdot e^x \\
&= -\frac{e^x \sin(e^x)}{\cos(e^x)} \\
&= -e^x \tan(e^x)
\end{align*}
$$
:::

The chain rule is pervasive in CCD because system performance depends on design variables through nested models: geometry changes mass and stiffness, those quantities change dynamics, and the dynamics change the objective and constraints. [This calculus reference](https://math.libretexts.org/Bookshelves/Calculus/Calculus_(OpenStax)/03%3A_Derivatives/3.06%3A_The_Chain_Rule) contains additional examples.

:::{tip} Activity 0.2.2
:class: dropdown
:name: derivatives-activity-2

Note that the activities in this section are quite challenging, so make sure you've attempted and fully understood the examples above first.

**Activity 2.1**

A useful smooth transition function for actuator models, switching approximations, and differentiable simulation is the **sigmoid function**, defined as:

$$
\sigma(x) = \frac{1}{1 + e^{-x}}
$$

$\sigma(x)$ has a nice S-shape, and is used for predicting probabilities.

Find the derivative of $\sigma(x)$, and show that it satisfies the following property:

$$
\frac{\text{d}}{\text{d}x} \sigma(x) = \sigma(x) (1 - \sigma(x))
$$

**Activity 2.2**

Find the derivative of each of the following functions:

- $f(x) = \sqrt{\sin(4\pi x)}$
- $g(x) = (2x + 1)^{3x}$ (Hint: Start by taking the natural logarithm of both sides, then take the derivative of both sides.)

**Activity 2.3**

Suppose $x$ and $y$ satisfy the following relationship:

$$x^2 = y^3 - 11$$

1. Find an expression for $\frac{\text{d}y}{\text{d}x}$ that involves **both** $x$ and $y$. To do this, don't solve for $y$ in terms of $x$ – instead, take the derivative of both sides of the equation with respect to $x$, use the power and chain rules, and re-arrange to isolate $\frac{\text{d}y}{\text{d}x}$.
2. Find the slope of the tangent line to the curve at the point $(x, y) = (4, 3)$.
:::

---

## Optimization

### Maxima and Minima

As stated at the start of this section, calculus is a tool for optimization – that is, finding the inputs that maximize or minimize a function. Let's be more precise about what we mean by "maximize" and "minimize".

Consider $f(x) = \frac{1}{4}x^4 + \frac{1}{3} x^3 - x^2 + 2$, shown below.

In [ ]:
from utils import plot_functions
import plotly
plotly.io.renderers.default = 'notebook'

f_list = [lambda x: (1/4)*x**4 + (1/3)*x**3 - x**2 + 2]
f_titles = ['f(x)=x^4/4+x^3/3-x^2+2']
crit_x = [-2, 0, 1]
crit_labels = ['global minimum', 'local maximum', 'local minimum']
x_range = (-5, 5)
y_range = (-2, 6)

fig = plot_functions(f_list, f_titles, crit_x, x_range, y_range)
fig.update_layout(width=600, height=400)

# Annotate f(x) for x in crit_x with the label (x, f(x))
f = lambda x: (1/4)*x**4 + (1/3)*x**3 - x**2 + 2
for i, x in enumerate(crit_x):
    y = f(x)
    fig.add_annotation(
        x=x,
        y=y + 0.1 * (1 if i % 2 == 0 else -1),
        text=crit_labels[i],
        showarrow=True,
        arrowhead=1,
        ax=0,
        ay=[-50, 40, -50][i],
        bgcolor="white",
        bordercolor="black",
        borderwidth=1
    )

fig.show(scale=4)

Where is $f(x)$ maximized and minimized?

- At $x=-2$, $f(x)$ is less than it is at all other inputs. This is means $(-2, f(-2))$ is a **global minimum**.
- At $x=0$, $f(x)$ _looks like_ it is greater than all other inputs, but only if you restrict your attention to points near $x=0$. This means $(0, f(0))$ is a **local maximum**. $(0, f(0))$ is not a **global maximum** because there are plenty of points where $f(x) > f(0)$ – they are just not immediately adjacent to $x=0$.
- Similarly, $(1, f(1))$ is a **local minimum**.

$f(x)$ does not have a global maximum, since $f(x)$ approaches infinity as $x$ increases beyond $x=1$ or decreases beyond $x=-2$. If we were to restrict the domain (or set of possible inputs) of $f(x)$ to, say, $[-3, 3]$, then there would be a global maximum at $x=3$.

Note that we usually care more about the inputs that maximize or minimize a function, rather than the actual values of the function at those inputs. In the above example, the fact that $x=-2$ is a global minimum is important; the fact that $f(-2) = -\frac{2}{3}$ is not _as_ important.

To recap:

:::{note} Definition: Maxima and Minima

Suppose $f: \mathbb{R} \to \mathbb{R}$ is a function.

On maxima:
- A function $f: \mathbb{R} \to \mathbb{R}$ has a **local maximum** at $x=x^*$ if $f(x^*) \geq f(x)$ for all $x$ in some interval around $x^*$.
- A function $f: \mathbb{R} \to \mathbb{R}$ has a **global maximum** at $x=x^*$ if $f(x^*) \geq f(x)$ for all $x$ in the domain of $f$.

Intuitively, a local maximum is a point where the function is higher than all nearby points, and a global maximum is a point where the function is higher than all points in the domain.

On minima:
- A function $f: \mathbb{R} \to \mathbb{R}$ has a **local minimum** at $x=x^*$ if $f(x^*) \leq f(x)$ for all $x$ in some interval around $x^*$.
- A function $f: \mathbb{R} \to \mathbb{R}$ has a **global minimum** at $x=x^*$ if $f(x^*) \leq f(x)$ for all $x$ in the domain of $f$.

:::

Note that maxima is the plural of maximum, minima is the plural of minimum, and extrema refers to both maxima and minima.

Below, you'll find several other examples of functions with varying amounts and types of extrema. Play close attention to the relationship between the two functions in the second row, $\color{#d81b60}h(x)$ and $\color{#004d40}k(x)$. $\color{#004d40}k(x)$ results from stretching $\color{#d81b60}h(x)$ vertically and shifting it up vertically, and has the same extrema.

In [ ]:
from utils import plot_functions_grid
import numpy as np

import plotly.graph_objects as go

f_0 = lambda x: x**3 + 1
f_1 = lambda x: -x**2
f_2 = lambda x: x**3 - 3*x
f_3 = lambda x: (1 / 4) * f_2(x) + 3
f_4 = lambda x: x**4 - 4*x**2
f_5 = lambda x: np.exp(x)
f_list = [f_0, f_1, f_2, f_3, f_4, f_5]
f_titles = [r"$f(x)=x^3+1$", 
            r"$g(x)=-x^2$",
            r"$h(x)=x^3-3x$", 
            r"$k(x)=\frac{1}{4}(x^3-3x)+3 = \frac{1}{4}{\color{#d81b60}h\color{#d81b60}(\color{#d81b60}x\color{#d81b60})} + 3$",
            r"$l(x)=x^4-4x^2$",
            r"$m(x)=e^x$"]
x_range = (-4, 4)
y_range = (-6, 6)

# Compute extrema for each function (within x_range)
extrema = [
    # f_0: y = x^3 + 1, monotonic, no local/global extrema
    [("no extrema!", 1, -1)],
    # f_1: y = -x^2, global max at x=0, global min at x=-4 and x=4 (endpoints)
    [("global maximum", 0, f_1(0)), ("global minimum", -4, f_1(-4)), ("global minimum", 4, f_1(4))],
    # f_2: y = x^3 - 3x, local max at x=-1, local min at x=1
    [("local maximum", -1, f_2(-1)), ("local minimum", 1, f_2(1))],
    # f_3
    [("local maximum", -1, f_3(-1)), ("local minimum", 1, f_3(1)), (r"$\text{extrema at same positions as in } h(x)=x^3-3x$", 1, -1)],
    # f_4: y = x^4 - 4x^2, local max at x=0, local min at x=-2,2
    [("local maximum", 0, f_4(0)), ("local minimum", -np.sqrt(2), f_4(-np.sqrt(2))), ("local minimum", np.sqrt(2), f_4(np.sqrt(2)))],
    # f_5: y = exp(x), global min at x=-4 (endpoint), no max
    [("no extrema!", 1, -1), ("approaches 0 but never reaches it", -2, 1)],
]

fig = plot_functions_grid(f_list, f_titles, x_range=x_range, y_range=y_range, rows=3, cols=2)

# Annotate extrema
for i, extrema_list in enumerate(extrema):
    for label, x_val, y_val in extrema_list:
        # Add annotation
        fig.add_annotation(
            x=x_val, y=y_val + 0.2 * (-1 if "min" in label else 1),
            text=label,
            showarrow=True if "mum" in label else False,
            arrowhead=1,
            ax=0,
            ay=20 if "min" in label else -20,
            bgcolor="white",
            bordercolor="black",
            borderwidth=1,
            font=dict(size=12),
            xref=f"x{i+1}",
            yref=f"y{i+1}",
        )
        # Add marker for actual extrema points
        if "no extrema" not in label and "approaches" not in label:
            fig.add_trace(
                go.Scatter(
                    x=[x_val],
                    y=[y_val],
                    mode='markers',
                    marker=dict(
                        color=['#3d81f6', 'orange', '#d81b60', '#004d40', '#6f42c1', '#20c997'][i],
                        size=6,
                        symbol='circle'
                    ),
                    showlegend=False
                ),
                row=(i // 2) + 1,
                col=(i % 2) + 1
            )
fig.update_layout(showlegend=False, width=700, height=900).show(renderer='notebook')

:::{tip} Activity 0.2.3
:class: dropdown
:name: derivatives-activity-3

Let $q(x) = 8x^4 - 4x$. $q(x)$ has a global minimum at $\left(\frac{1}{2}, -\frac{3}{2}\right)$.

For each of the following functions, find all extrema, and specify whether each extremum is a local maximum, global maximum, local minimum, or global minimum. Make sure to specify both the $x$-values and the $y$-values of each extremum.

1. $f(x) = 2q(x) + 10$
1. $g(x) = -10q(x)$
1. $h(x) = \big| q(x) \big|$
1. Finding the extrema of $l(x) = q(x)^2$ is a bit more complicated than in the examples above. Why?

:::

### Critical Points

How do we actually find the critical points of a function, especially when we can't graph the function? The derivative plays a crucial role.

:::{note} Definition: Critical Point

A **critical point** of a function $f: \mathbb{R} \to \mathbb{R}$ is a point $(a, f(a))$ where $\frac{\text{d}f}{\text{d}x}(a)=0$ or $\frac{\text{d}f}{\text{d}x}(a)$ is undefined.
:::

Let's revisit the function $f(x) = \frac{1}{4}x^4 + \frac{1}{3} x^3 - x^2 + 2$, shown in <span style="color: #3d81f6; font-weight: bold;">blue</span> along with its derivative, $\frac{\text{d}f}{\text{d}x} = x^3 + x^2 - 2x$, shown in <span style="color: orange; font-weight: bold;">orange</span>.

In [ ]:
from utils import plot_functions
import plotly
plotly.io.renderers.default = 'notebook'

f_list = [lambda x: (1/4)*x**4 + (1/3)*x**3 - x**2 + 2, lambda x: x**3 + x**2 - 2*x]
f_titles = ['f(x)=x^4/4+x^3/3-x^2+2', 'f\'(x)=x^3+x^2-2x']
crit_x = [-2, 0, 1]
x_range = (-5, 5)
y_range = (-2, 6)

fig = plot_functions(f_list, f_titles, crit_x, x_range, y_range)

fig.add_annotation(
    x=-2.7, y=2.2,
    text=r"$\text{Original Function, } f(x)$",
    showarrow=True,
    arrowhead=1,
    ax=60,
    ay=-50,
    font=dict(color="#3d81f6"),
    bordercolor="#3d81f6",
    borderwidth=1,
    bgcolor="white"
)

fig.add_annotation(
    x=1.3, y=0.8,
    text=r"$\text{Derivative, } \frac{\text{d}f}{\text{d}x}$",
    showarrow=True,
    arrowhead=1,
    ax=40,
    ay=40,
    font=dict(color="orange"),
    bordercolor="orange",
    borderwidth=1,
    bgcolor="white"
)

fig.update_layout(width=600, height=400).show(scale=3)

You should notice that the <span style="color: orange; font-weight: bold;">derivative is 0</span> at all three extrema we identified earlier – the global minimum at $x=-2$, the local maximum at $x=0$, and the local minimum at $x=1$. Intuitively, the derivative is 0 at a maximum or minimum because the tangent lines at these points are horizontal (with slope 0), as the function is neither increasing nor decreasing at these points. 

In the region between $x=-2$ and $x=0$, the derivative is positive, meaning the function is increasing.

Solving for the inputs that make the derivative 0 – i.e., finding the critical points – is a necessary, but not sufficient, step. If all we know is that the derivative is 0 at a point, we don't know whether the point is a maximum or minimum. It may not be either, such as in the case of $f(x) = x^3$, which has a critical point at $x = 0$ that is neither a maximum nor a minimum.

### Second Derivatives

To be able to determine whether a critical point of $f(x)$ is a maximum or minimum, we need to look at the **second derivative** of $f(x)$. If the (first) derivative of $f(x)$ is a function that describes the rate at which $f(x)$ is changing, the second derivative – denoted $\frac{\text{d}^2f}{\text{d}x^2}$ – is a function that describes the rate at which the derivative is changing.

Physics provides us with an analogy that helps us understand the role of the second derivative. Suppose you're driving down a straight road, and $s(t)$ is your position on the road at time $t$, relative to your starting point (so a negative value of $s(t)$ means you've moved backwards).

Then, $v(t) = \frac{\text{d}s}{\text{d}t}$ is your velocity (the rate at which your position is changing) and $a(t) = \frac{\text{d}^2s}{\text{d}t^2}$ is your acceleration (the rate at which your velocity is changing).

- If $\frac{\text{d}s}{\text{d}t} > 0$ and $\frac{\text{d}^2s}{\text{d}t^2} = 0$, you are moving forward at a constant speed (say, on cruise control).
- If $\frac{\text{d}s}{\text{d}t} > 0$ and $\frac{\text{d}^2s}{\text{d}t^2} > 0$, you are moving forward and your speed is increasing (you are accelerating).
- If $\frac{\text{d}s}{\text{d}t} > 0$ and $\frac{\text{d}^2s}{\text{d}t^2} < 0$, you are moving forward, but your speed is decreasing, and **eventually, your car will come to a halt**.
- Cases where $\frac{\text{d}s}{\text{d}t} < 0$ correspond to driving backwards!

:::{tip} Activity 0.2.4
:class: dropdown
:name: derivatives-activity-4

Give real-world examples (similar to those provided above) for each of the following scenarios in the context of the physics analogy:

- $\frac{\text{d}s}{\text{d}t} < 0$ and $\frac{\text{d}^2s}{\text{d}t^2} > 0$
- $\frac{\text{d}s}{\text{d}t} < 0$ and $\frac{\text{d}^2s}{\text{d}t^2} < 0$
- $\frac{\text{d}s}{\text{d}t} = 0$ and $\frac{\text{d}^2s}{\text{d}t^2} < 0$
:::

Let's put this in the context of our running example, $f(x) = \frac{1}{4}x^4 + \frac{1}{3} x^3 - x^2 + 2$. The second derivative of $f(x)$ is:

$$
\begin{align*}
\frac{\text{d}^2f}{\text{d}x^2} &= \frac{\text{d}}{\text{d}x} \left( \frac{\text{d}}{\text{d}x} \left( \frac{1}{4}x^4 + \frac{1}{3} x^3 - x^2 + 2 \right) \right) \\
&= \frac{\text{d}}{\text{d}x} \left( \underbrace{x^3 + x^2 - 2x}_{\text{first derivative of } f(x)} \right) \\
&= 3x^2 + 2x - 2
\end{align*}
$$

The second derivative, $\frac{\text{d}^2f}{\text{d}x^2}$, is a function, not a number. What does the <span style="color: #d81b60; font-weight: bold;">second derivative</span> look like, relative to the <span style="color: #3d81f6; font-weight: bold;">original function</span> and <span style="color: orange; font-weight: bold;">first derivative</span>?

In [ ]:
from utils import plot_functions_grid

f_list = [
    lambda x: (1/4)*x**4 + (1/3)*x**3 - x**2 + 2,
    lambda x: x**3 + x**2 - 2*x,
    lambda x: 3*x**2 + 2*x - 2
]
f_titles = [
    r'$\text{Original Function, } f(x)$',
    r'$\text{First Derivative, } \frac{\text{d}f}{\text{d}x}$',
    r'$\text{Second Derivative, } \frac{\text{d}^2f}{\text{d}x^2}$'
]
crit_x = [-2, 0, 1]
x_range = (-6, 6)
y_range = (-3, 7)

fig = plot_functions_grid(
    f_list=f_list,
    f_titles=f_titles,
    rows=1,
    cols=3,
    x_range=x_range,
    y_range=y_range,
    title=None,
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title=''
)

# Mark critical points only on the first two plots (original and first derivative)
import numpy as np
for i in range(3):
    y_crit = f_list[i](np.array(crit_x))
    fig.add_trace(
        go.Scatter(
            x=crit_x, y=y_crit, mode='markers',
            marker=dict(color=['#3d81f6', 'orange', '#d81b60', '#004d40', '#6f42c1', '#20c997'][i], size=8),
            showlegend=False
        ),
        row=1, col=i+1
    )

# Add extra top margin so titles are not cut off
fig.update_layout(width=900, height=350, showlegend=False, margin=dict(t=50, l=20, r=20, b=0))
fig.show(renderer='notebook')

$f(x)$ is a polynomial of degree 4, $\frac{\text{d}f}{\text{d}x}$ is a polynomial of degree 3, and $\frac{\text{d}^2f}{\text{d}x^2}$ is a polynomial of degree 2 – the degree drops by one each time, as a consequence of the power rule.

Recall that $f(x)$ has critical points at $x=-2$, $x=0$, and $x=1$, which we've highlighted in all three plots above. Our goal is to determine an algebraic approach for determining whether these points are maxima, minima, or neither; we shouldn't rely on the graph of $f(x)$ alone, since we won't always be able to see its graph.

At all three of these points, the first derivative is 0, meaning that the function is neither increasing nor decreasing at these points. But the second derivative $\frac{\text{d}^2f}{\text{d}x^2} = 3x^2 + 2x - 2$ gives us additional information:

- At $x = -2$, $\frac{\text{d}^2f}{\text{d}x^2}(-2) = 6$, which is **positive**. So, at $x = -2$, $f(x)$ is neither increasing nor decreasing, but is also "_speeding up_", since the second derivative is positive. So, as we move to the right of $x=-2$, the slope of the tangent line will increase, causing the function to increase. If the function increases to the right of $x=-2$, then $x=-2$ must correspond to a **local minimum** of $f(x)$.

- At $x = 0$, $\frac{\text{d}^2f}{\text{d}x^2}(0) = -2$, which is **negative**. So, at $x = 0$, $f(x)$ is neither increasing nor decreasing, but is also "_slowing down_". So, as we move to the right of $x=0$, the slope of the tangent line will decrease, causing the function to decrease. If the function decreases to the right of $x=0$, then $x=0$ must correspond to a **local maximum** of $f(x)$.

- At $x = 1$, $\frac{\text{d}^2f}{\text{d}x^2}(1) = 3$ is **positive**, which, using the logic from the $x = -2$ case, means that $x = 1$ also corresponds to a **local minimum** of $f(x)$.

### Convexity

The sign of the second derivative is useful for more than just determining whether a critical point is a local maximum or minimum. Below, we've plotted $f(x)$, along with annotations for the regions where the second derivative is positive and negative.

In [ ]:
from utils import plot_functions
import numpy as np
import plotly.graph_objects as go

# Define the original function and its second derivative
f = lambda x: (1/4)*x**4 + (1/3)*x**3 - x**2 + 2
second_derivative = lambda x: 3*x**2 + 2*x - 2

x_range = (-6, 6)
y_range = (-3, 7)

# Find inflection points: roots of the second derivative
coeffs = [3, 2, -2]
inflection_points = np.roots(coeffs)
inflection_points = [float(x) for x in inflection_points if np.isreal(x) and x_range[0] <= x <= x_range[1]]
inflection_points.sort()

# Plot just f(x)
fig = plot_functions(
    f_list=[f],
    f_titles=[r'$\text{Original Function, } f(x)$'],
    x_range=x_range,
    y_range=y_range,
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title=''
)

# Shade regions: outside inflection points = pink, between = orange
x0, x1 = inflection_points
fig.add_vrect(
    x0=x_range[0], x1=x0,
    fillcolor="pink", opacity=0.15, line_width=0, layer="below"
)
fig.add_vrect(
    x0=x0, x1=x1,
    fillcolor="orange", opacity=0.15, line_width=0, layer="below"
)
fig.add_vrect(
    x0=x1, x1=x_range[1],
    fillcolor="pink", opacity=0.15, line_width=0, layer="below"
)

# Add vertical dotted lines at inflection points
for x0 in inflection_points:
    fig.add_vline(
        x=x0,
        line=dict(color="black", width=2, dash="dot")
    )

# Add text annotations for the three regions based on the sign of the second derivative
region_xs = [
    (x_range[0] + inflection_points[0]) / 2 - 0.75,
    (inflection_points[0] + inflection_points[1]) / 2,
    (inflection_points[1] + x_range[1]) / 2
]
region_ys = [5, 5, 5]

region_signs = []
for x in region_xs:
    val = second_derivative(x)
    if val > 0:
        region_signs.append("second<br>derivative<br><span style='color:#d81b60'>positive</span>")
    elif val < 0:
        region_signs.append("second<br>derivative<br><span style='color:orange'>negative</span>")
    else:
        region_signs.append("second<br>derivative<br>zero")

for x, y, text in zip(region_xs, region_ys, region_signs):
    fig.add_annotation(
        x=x,
        y=y,
        text=text,
        showarrow=False,
        font=dict(size=14, color="black"),
        align="center"
    )

# Hide x and y axis lines
fig.update_xaxes(showline=False, zeroline=False)
fig.update_yaxes(showline=False, zeroline=False)

# Add happy and sad face emojis in the "cups" (minima) and "bowls" (maxima)
# For this quartic, local minima at x = -2 and x = 1, local maximum at x = 0
minima_xs = [-2, 1]
maxima_xs = [0]
for x in minima_xs:
    y = f(x)
    fig.add_annotation(
        x=x+0.05,
        y=y+0.75,
        text="😊",
        showarrow=False,
        font=dict(size=24)
    )
for x in maxima_xs:
    y = f(x)
    fig.add_annotation(
        x=x,
        y=y-0.75,
        text="😢",
        showarrow=False,
        font=dict(size=24)
    )

fig.update_layout(width=600, height=350, showlegend=False, margin=dict(t=5, l=20, r=20, b=0))
fig.show(renderer='notebook')

When the second derivative is positive, the function is **concave opening up**, also known as **convex**. You should think of convex functions as "bowl-shaped" or "smiling". When the second derivative is negative, the function is **concave opening down**, or simply **concave**; the equivalent analogy is that concave down regions are "upside-down bowls" or "sad faces".

From the perspective of finding local mimina, if a function is concave up at a critical point, then we must be at the bottom of a bowl – a local minimum – and if a function is concave down at a critical point, we must be at the top of a hill, corresponding to a local maximum.

If a function is convex across its entire domain—unlike the example above, but like $f(x)=x^2$—then any local minimum is also a global minimum. Convexity is central to optimization, while most realistic CCD problems remain nonlinear and nonconvex because plant and control decisions interact through the dynamics.

The points at which the second derivative is 0 are called **inflection points**. $f(x)$ has two inflection points, marked by vertical dotted lines above, roughly at $x = -1.22$ and $x = 0.55$. These are the roots of the quadratic equation $\frac{\text{d}^2f}{\text{d}x^2} = 3x^2 + 2x - 2 = 0$.

We've implictly used a **second derivative test** for determining whether a critical point is a local maximum or minimum:

:::{note} Definition: Second Derivative Test

Suppose $a$ is a critical point of $f(x)$, i.e. $\frac{\text{d}}{\text{d}x}f(a)=0$ or $\frac{\text{d}}{\text{d}x}f(a)$ is undefined. Then, there are 3 possibilities:

- If $\frac{\text{d}^2f}{\text{d}x^2}(a)>0$, then $f(x)$ is concave opening up (i.e. convex) at $x = a$, and $x = a$ is a local **minimum**.
- If $\frac{\text{d}^2f}{\text{d}x^2}(a)<0$, then $f(x)$ is concave opening down at $x = a$, and $x = a$ is a local **maximum**.
- If $\frac{\text{d}^2f}{\text{d}x^2}(a)=0$, then $x = a$ is an inflection point, and the second derivative test is inconclusive.
:::

Again, the second derivative test only tries to tell us whether critical points are local maxima or minima; it **does not** tell us whether they are global maxima or minima.

Let's look at another example, particularly one where the second derivative test is inconclusive. Consider $f(x) = x^2 \sin(x)$, shown below.

In [ ]:
from utils import plot_functions
import numpy as np

f_list = [
    lambda x: (x ** 2) * np.sin(x) ,
    # lambda x: x**3 + x**2 - 2*x,
    # lambda x: 3*x**2 + 2*x - 2
]
f_titles = [
    r'$f(x) = x^2 \sin(x)$',
    r'$\text{First Derivative, } \frac{\text{d}f}{\text{d}x}$',
    r'$\text{Second Derivative, } \frac{\text{d}^2f}{\text{d}x^2}$'
][:1]
# crit_x = [-2, 0, 1]
x_range = (-2 * np.pi, 2 * np.pi)
y_range = (-25, 25)

fig = plot_functions(
    f_list=f_list,
    f_titles=f_titles,
    x_range=x_range,
    y_range=y_range,
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title=''
)

# Mark critical points only on the first two plots (original and first derivative)
# import numpy as np
# for i in range(len(f_list)):
#     y_crit = f_list[i](np.array(crit_x))
#     fig.add_trace(
#         go.Scatter(
#             x=crit_x, y=y_crit, mode='markers',
#             marker=dict(color=['#3d81f6', 'orange', '#d81b60', '#004d40', '#6f42c1', '#20c997'][i], size=8),
#             showlegend=False
#         ),
#         row=1, col=i+1
#     )

# Add extra top margin so titles are not cut off
fig.update_layout(width=500, height=350, showlegend=False, margin=dict(t=50, l=20, r=20, b=0))
fig.show(renderer='notebook')

$f(x) = x^2 \sin(x)$, like $\sin(x)$, is oscillatory, and has no global extrema; see [**here**](https://www.desmos.com/calculator/nvzesx95xo) for a larger graph of it. Above, we've plotted $f(x)$ within the domain $[-2\pi, 2\pi]$, and we see several local maxima and minima.

The first and second derivatives of $f(x) = x^2 \sin(x)$ are given by:

$$
\begin{aligned}
\frac{\text{d}f}{\text{d}x} &= x^2 \left( \frac{\text{d}}{\text{d}x} \sin(x) \right) + \left( \frac{\text{d}}{\text{d}x} x^2 \right) \sin(x) \\
&= x^2 \cos(x) + 2x \sin(x) \\ \\
\frac{\text{d}^2f}{\text{d}x^2} &= \frac{\text{d}}{\text{d}x} \left( x^2 \cos(x) + 2x \sin(x) \right) \\
&= x^2 \left( \frac{\text{d}}{\text{d}x} \cos(x) \right) + \left( \frac{\text{d}}{\text{d}x} x^2 \right) \cos(x) + 2x \left( \frac{\text{d}}{\text{d}x} \sin(x) \right) + \left( \frac{\text{d}}{\text{d}x} 2x \right) \sin(x) \\
&= - x^2 \sin(x) + 2x \cos(x) + 2x \cos(x) + 2 \sin(x) \\
&= 2x \cos(x) - (x^2 - 2) \sin(x) \\
\end{aligned}
$$

Solving for the critical points of $f(x)$ by setting $\frac{\text{d}f}{\text{d}x} = x^2 \cos(x) + 2x \sin(x) = 0$ is no easy task, as there are infinitely many solutions, most of which cannot be solved for by hand. Numerical optimization methods such as gradient descent can approximate solutions to $\frac{\text{d}f}{\text{d}x} = 0$. There are also infinitely many inflection points, since $\frac{\text{d}^2f}{\text{d}x^2} = 0$ has infinitely many solutions, meaning that there are many regions where $f(x)$ is concave up and many others where it is concave down.

However, one critical point is easy to spot: $x = 0$. At $x = 0$, the derivative is 0:

$$\frac{\text{d}f}{\text{d}x}(0) = 0^2 \cos(0) + 2 \cdot 0 \cdot \sin(0) = 0$$

$x = 0$ is also an inflection point, since the second derivative is also 0:

$$\frac{\text{d}^2f}{\text{d}x^2}(0) = 2 \cdot 0 \cdot \cos(0) - (0^2 - 2) \sin(0) = 0$$

To be clear, not every inflection point is a critical point, and not every critical point is an inflection point; $x = 0$ just happens to be both.

If we look at the graph of $f(x)$ near $x = 0$, we'll see that $x = 0$ corresponds to neither a local maximum nor a local minimum, but rather, a region where $f(x)$ is very flat. If we weren't able to graph $f(x)$, we could try and determine its behavior around $(0, f(0))$ by looking at points immediately to the left and right of $x = 0$ – say, $(0.001, f(0.001))$ and $(-0.001, f(-0.001))$. If $f(0.001) > f(0)$ and $f(-0.001) > f(0)$, then $x = 0$ would be a local minimum (but that's not the case here).

:::{tip} Activity 0.2.5
:class: dropdown
:name: derivatives-activity-5

**Activity 5.1**

Let $f(x) = x \log(x^2)$, where $\log( \cdot )$ is the natural logarithm.

1. Find the critical points of $f(x)$, and determine whether they are local maxima, minima, or neither.
2. Find the inflection points of $f(x)$, and use them to sketch a possible graph of $f(x)$.

**Activity 5.2**

Let $g(3) = 10$, $\frac{\text{d}g}{\text{d}x}(3) = -2$, and $\frac{\text{d}^2g}{\text{d}x^2}(3) = 1$.

1. Describe the behavior of $g(x)$ near $x = 3$.
2. The Taylor series of a function allows us to approximate the value of a function near a point $x = a$, given the value of the function and its derivatives at $x = a$. The Taylor series of an arbitrary function $f(x)$ around $x = a$ is given by:
    $$
    f(x) \approx f(a) + \left(\frac{\text{d}f}{\text{d}x}(a) \right)(x - a) + \frac{1}{2}\left(\frac{\text{d}^2f}{\text{d}x^2}(a) \right)(x - a)^2 + \frac{1}{6} \left(\frac{\text{d}^3f}{\text{d}x^3}(a) \right)(x - a)^3 + \frac{1}{24} \left(\frac{\text{d}^4f}{\text{d}x^4}(a) \right)(x - a)^4 + \cdots
    $$

    Note that this is an infinite series; the more terms we use, the more accurate the approximation.

    Use the Taylor series to approximate the value of $g(3.1)$, using only the information provided. You'll only be able to use the first 3 terms of the Taylor series.

**Activity 5.3**

Given that $\frac{\text{d}^2h}{\text{d}x^2} = 2x(x - 3)(x + 1)$, sketch a possible graph of $h(x)$.

---

## Continuity and Differentiability

Finally, I'll remark that we've presented derivatives, extrema, and optimization all in the most ideal setting: where the functions we're working with are continuous and differentiable. A function is **continuous** if its graph can be drawn without lifting a pen; any point where the graph has a "jump" or "break" is a discontinuity. (Of course, there is a more formal definition of continuity, but this is a good enough illustration for now.) A function is **differentiable** if its derivative exists everywhere; otherwise, there exist some points at which the derivative does not exist.

CCD models often contain nonsmooth elements such as saturation, contact, absolute-value penalties, lookup tables, mode switches, and state resets. These features can degrade derivative accuracy or violate a solver's assumptions, so it is important to recognize continuity and differentiability before optimizing.

**Example 1**: $\color{#3d81f6} f(x) = |x|$

In [ ]:
import numpy as np
from utils import plot_functions

f = lambda x: np.abs(x)
x_range = (-10, 10)
y_range = (-0.5, 10)

fig_f = plot_functions(
    [f],
    [r'$f(x)=|x|$'],
    x_range=x_range,
    y_range=y_range,
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title=''
)
fig_f.add_annotation(
    x=0, y=2,
    text="not differentiable<br>at x=0",
    showarrow=True,
    arrowhead=1,
    ax=0, ay=-60,
    font=dict(size=12),
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)
fig_f.update_layout(width=600, height=300, margin=dict(t=0, l=0, r=0, b=0)).show(renderer='notebook')

$\color{#3d81f6} {f(x) = |x|}$ is continuous everywhere, as intuitively, we can draw its graph without lifting our pen. It is differentiable everywhere **except** at $x = 0$; the reason it is not differentiable at $x = 0$ is that the slopes approaching it from the left ($-1$) and right ($1$) are different, and in order for a derivative at $x = a$ to exist, the limit of the slopes approaching $a$ from the left and right must be the same.

$$
\frac{\text{d}{\color{#3d81f6} f}}{\text{d}x} = \begin{cases} -1 & x < 0 \\ 1 & x > 0 \\ \text{undefined} & x = 0 \end{cases}
$$

<br>

**Example 2**: $\color{orange} g(x) = \begin{cases} x^3-3x + 4 & x \neq 2 \\ 1 & x = 2 \end{cases}$

In [ ]:
import numpy as np
from utils import plot_functions

g = lambda x: np.where(x == 2, 1, x**3 - 3*x + 4)
x_range = (-10, 10)
y_range = (-0.5, 10)

fig_g = plot_functions(
    [g],
    [r'$g(x)=\begin{cases} x^3-3x + 4 & x \neq 2 \\ 1 & x = 2 \end{cases}$'],
    x_range=x_range,
    y_range=y_range,
    # title=r'$g(x)=\begin{cases} x^3-3x + 4 & x \neq 2 \\ 1 & x = 2 \end{cases}$',
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title='',
    color_just_one=1
)
fig_g.add_trace(
    dict(
        type='scatter',
        x=[2], y=[1],
        mode='markers',
        marker=dict(color='orange', size=10, symbol='circle'),
        showlegend=False
    )
)
fig_g.add_trace(
    dict(
        type='scatter',
        x=[2], y=[g(2.0001)],
        mode='markers',
        marker=dict(color='white', size=10, symbol='circle', line=dict(color='orange', width=2)),
        showlegend=False
    )
)
fig_g.add_annotation(
    x=2.5, y=6.25,
    text="discontinuous<br>at x=2",
    showarrow=True,
    arrowhead=1,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    ax=40, ay=-40,
    font=dict(size=12),
)
fig_g.update_layout(width=700, height=400, margin=dict(t=0, l=0, r=0, b=0)).show(renderer='notebook')

$\color{orange} g(x) = \begin{cases} x^3-3x + 4 & x \neq 2 \\ 1 & x = 2 \end{cases}$ is continuous and differentiable everywhere, except at $x = 2$, where it is neither. 

$$
\frac{\text{d}{\color{orange} g}}{\text{d}x} = 
\begin{cases}
3x^2 - 3 & x \neq 2 \\
\text{undefined} & x = 2
\end{cases}
$$

<br>

**Example 3**: $\color{#d81b60} h(x)=\begin{cases} x^2 + \frac{1}{2}x + 1 & x < 0 \\ \sqrt{x + 1} & x \geq 0 \end{cases}$

In [ ]:
# --- Cell 3: Plot for h(x) ---
import numpy as np
from utils import plot_functions

h = lambda x: np.where(x < 0, x ** 2 + (1/2)*x + 1, np.sqrt(np.abs(x) + 1))
x_range = (-10, 10)
y_range = (-0.5, 10)

fig_h = plot_functions(
    [h],
    [r'$h(x)=\begin{cases} x^2 + \frac{1}{2}x + 1 & x < 0 \\ \sqrt{x + 1} & x \geq 0 \end{cases}$'],
    x_range=x_range,
    y_range=y_range,
    # title=r'$h(x)=\begin{cases} x^2 + \frac{1}{2}x + 1 & x < 0 \\ \sqrt{x + 1} & x \geq 0 \end{cases}$',
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title='',
    color_just_one=2
)
fig_h.add_annotation(
    x=4, y=5,
    text="differentiable<br>(and continuous)<br>everywhere",
    showarrow=False,
    arrowhead=1,
    ax=40, ay=-40,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)
fig_h.update_layout(width=600, height=300, margin=dict(t=0, l=0, r=0, b=0)).show(renderer='notebook')

$\color{#d81b60} h(x)=\begin{cases} x^2 + \frac{1}{2}x + 1 & x < 0 \\ \sqrt{x + 1} & x \geq 0 \end{cases}$ is continuous and differentiable everywhere, despite being a piecewise function. Its individual pieces are continuous, and the entire function is continuous because the "left" and "right" functions at the connection point of $x = 0$ have the same value.

$$
\frac{\text{d}{\color{#d81b60} h}}{\text{d}x} = \begin{cases} 2x + \frac{1}{2} & x < 0 \\ \frac{1}{2\sqrt{x + 1}} & x \geq 0 \end{cases}
$$

Since the two piecewise derivatives agree at $x = 0$, $\color{#d81b60}h(x)$ is differentiable at $x = 0$ (and across its entire domain).

<br>

**Example 4**: $\color{#004d40} k(x)=\begin{cases} -{(x+2)}^2+5 & x < -2 \\ \frac{1}{2}x+6 & x \geq -2,\, x \neq 4 \\ \text{undefined} & x=4 \end{cases}$

In [ ]:
import numpy as np
from utils import plot_functions

def k(x):
    out = np.where(x < -2, -(x + 2)**2 + 5, x / 2 + 6)
    out = np.where(x == 4, np.nan, out)
    return out

x_range = (-10, 10)
y_range = (-0.5, 10)

fig_k = plot_functions(
    [k],
    [r'$k(x)=\begin{cases} -{(x+2)}^2+5 & x < -2 \\ \frac{1}{2}x+6 & x \geq -2,\, x \neq 4 \\ \text{undefined} & x=4 \end{cases}$'],
    x_range=x_range,
    y_range=y_range,
    # title=r'$k(x)=\begin{cases} -{(x+2)}^2+5 & x < -2 \\ \frac{1}{2}x+6 & x \geq -2,\, x \neq 4 \\ \text{undefined} & x=4 \end{cases}$',
    show_axis_labels=True,
    xaxis_title='',
    yaxis_title='',
    color_just_one=3
)
fig_k.add_trace(
    dict(
        type='scatter',
        x=[4], y=[8],
        mode='markers',
        marker=dict(color='white', size=10, symbol='circle', line=dict(color='#004d40', width=2)),
        showlegend=False
    )
)
fig_k.add_annotation(
    x=-2.5, y=5,
    text="not differentiable<br>at x=-2",
    showarrow=True,
    arrowhead=1,
    ax=-40, ay=-40,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)
fig_k.add_annotation(
    x=4.2, y=7.5,
    text="discontinuous<br>at x=4",
    showarrow=True,
    arrowhead=1,
    ax=20, ay=40,
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
    font=dict(size=12),
)
fig_k.update_layout(width=600, height=300, margin=dict(t=0, l=0, r=0, b=0)).show(renderer='notebook')

$\color{#004d40} k(x)=\begin{cases} -{(x+2)}^2+5 & x < -2 \\ \frac{1}{2}x+6 & x \geq -2,\, x \neq 4 \\ \text{undefined} & x=4 \end{cases}$ is continuous everywhere, except at $x = 4$, where it has a "jump" and is neither continuous nor differentiable. But in addition, $\color{#004d40}k(x)$ is not differentiable at $x = -2$ because the slopes approaching it from the left and right are different. 

$$
\frac{\text{d}{\color{#004d40} k}}{\text{d}x} = \begin{cases} -2(x+2) & x < -2 \\ \text{undefined} & x = -2 \\ \frac{1}{2} & x > -2,\, x \neq 4 \\ \text{undefined} & x=4 \end{cases}
$$

An important point is that any function that is differentiable everywhere is also continuous everywhere; differentiability is a stronger condition than continuity. Plenty of functions are continuous but not differentiable, like $\color{#3d81f6} f(x) = |x|$ in Example 1.